#Makmore

it basically will train on a large dataset of random names then it predict / give you a new names which is somewhat similar to the training dataset


so,under the makemore is a character level language model. evry single word  given to it, it learns by going through the charachters of all those words. so basically it will predict characters and then eventually combining them to form word


Current language model neural net implemented :
1. bigram
2. Bag of words
3. mlp
4. rnn
5. gru
6. transformer

after learning this character level language model we will spend some time on word level so that we can generate huge documents then images/image text networks

but here we are going for character level lang. model

In [6]:
words = open("/content/drive/MyDrive/Andrej/names.txt",'r').read().splitlines()
#names.txt is the training dataset of random words and splitlines have converted that into list

In [7]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [8]:
len(words)

32033

In [9]:
min(len(w) for w in words) #shortest word len

2

In [10]:
max(len(w) for w in words) #longest word len

15

when we come to ch. level lanf. model its kind off very complex because each word have a lot of ch. and its hard to find a scheme to predict a new ch. based on a huge dataset

# Building Bigram model

in bigram lang. model we are always working with just two characters at a time.

So, we're only looking at one character that we are given, and then we are trying to predict the next character in the sequence e.g; what ch. are likely to follow 'a',etc

in this we basically forget about everything and just look at the prev. character and what to predict on the basis of that

so its a very weak and simple model but its a great place to start


In [15]:
#lets look at the bigram of out first word (emma) in the dataseet

for w in words[:1]:
  chs = ['<S>'] + list(w) + ['<E>'] # we have added s to know the starting ch. and e to know the endiong ch.
  for ch1,ch2 in zip(chs, chs[1:]):  # to know how we got the emmmma please chat gpt about how zip works
    print(ch1,  ch2)

# so ch1 and ch2 are the consecutive element in word 1

<S> e
e m
m m
m a
a <E>


now in order to learn the statistics about which characters are likely to follow other characters, the simplest way in the bigram lang. model is to simply do it bu counting.

so we are basically going to cout how often does any of these combinations occurs in the training set in these words.

so we're going to need some dictionary that is going maintaing some counts for every one of these bigrams

In [17]:
b = {} #dictionary
for w in words[:1]:
  chs = ['<S>'] + list(w) + ['<E>'] # we have added s to know the starting ch. and e to know the endiong ch.
  for ch1,ch2 in zip(chs, chs[1:]):  # to know how we got the emmmma please chat gpt about how zip works
    bigram = (ch1, ch2) # so bigram is the touple of ch1 and ch2
    b[bigram] = b.get(bigram, 0)+1 #bigram, 0 if bigram is not in the dictionary b we will return 0

In [18]:
b # this is telling how often does each bigram occurs

{('<S>', 'e'): 1, ('e', 'm'): 1, ('m', 'm'): 1, ('m', 'a'): 1, ('a', '<E>'): 1}

In [20]:
# lets run for the whole dataset

b = {} #dictionary
for w in words:
  chs = ['<S>'] + list(w) + ['<E>'] # we have added s to know the starting ch. and e to know the endiong ch.
  for ch1,ch2 in zip(chs, chs[1:]):  # to know how we got the emmmma please chat gpt about how zip works
    bigram = (ch1, ch2) # so bigram is the touple of ch1 and ch2
    b[bigram] = b.get(bigram, 0)+1 #bigram, 0 if bigram is not in the dictionary b we will return 0


b

{('<S>', 'e'): 1531,
 ('e', 'm'): 769,
 ('m', 'm'): 168,
 ('m', 'a'): 2590,
 ('a', '<E>'): 6640,
 ('<S>', 'o'): 394,
 ('o', 'l'): 619,
 ('l', 'i'): 2480,
 ('i', 'v'): 269,
 ('v', 'i'): 911,
 ('i', 'a'): 2445,
 ('<S>', 'a'): 4410,
 ('a', 'v'): 834,
 ('v', 'a'): 642,
 ('<S>', 'i'): 591,
 ('i', 's'): 1316,
 ('s', 'a'): 1201,
 ('a', 'b'): 541,
 ('b', 'e'): 655,
 ('e', 'l'): 3248,
 ('l', 'l'): 1345,
 ('l', 'a'): 2623,
 ('<S>', 's'): 2055,
 ('s', 'o'): 531,
 ('o', 'p'): 95,
 ('p', 'h'): 204,
 ('h', 'i'): 729,
 ('<S>', 'c'): 1542,
 ('c', 'h'): 664,
 ('h', 'a'): 2244,
 ('a', 'r'): 3264,
 ('r', 'l'): 413,
 ('l', 'o'): 692,
 ('o', 't'): 118,
 ('t', 't'): 374,
 ('t', 'e'): 716,
 ('e', '<E>'): 3983,
 ('<S>', 'm'): 2538,
 ('m', 'i'): 1256,
 ('a', 'm'): 1634,
 ('m', 'e'): 818,
 ('<S>', 'h'): 874,
 ('r', 'p'): 14,
 ('p', 'e'): 197,
 ('e', 'r'): 1958,
 ('r', '<E>'): 1377,
 ('e', 'v'): 463,
 ('v', 'e'): 568,
 ('l', 'y'): 1588,
 ('y', 'n'): 1826,
 ('n', '<E>'): 6763,
 ('b', 'i'): 217,
 ('i', 'g'): 428,


now its going to be significiantly convienent for us to store all this information ina 2d array instead of a dictionary, so we are going to store all these info in a 2d array

the rows are going to be the first character of the bigram and the columns are going to be the second charater and each entry in this 2d array will tell us how does that 2nd ch. follows the 1st  ch.

sp fpr this purpose the library we're going to use is that of pytorch and part of is the torch.tensor which allows us to create multi dimensional array and manipulate them very efficiently

In [21]:
import torch

In [22]:
a = torch.zeros((2,3))
a

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [24]:
a.dtype #by default their dtype

torch.float32

In [26]:
# 3but we want the dtype to be int

a = torch.zeros((3,5),dtype = torch.int32)
a

tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

tensors allows us to manipulate the individual tensors  very efficiently

In [27]:
# e.g

a[1,3] = 1
a

tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

for our purpose we have 25 ch. of alphabets and 2 special ch. < S > and < E >

so we want 28x28 array

In [28]:
N = torch.zeros((28,28), dtype = torch.int32)

now the problem is that we have characters that are strings but we have to index using integers as n is of int type
so, we need some kind of look up table from characters to integers.

So lets construct suxh a character array. And the we're going to do is we're going to take all the words in the db ang concat it to make a single word and we'are going to pass it to the set constructor as set only contains unique values. so the set of this will be all the lower case characters

In [29]:
set(''.join(words))

{'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z'}

In [31]:
#there should be 26 ch.

len(set(''.join(words)))

26

In [33]:
# but we want a list sorted in a-z

chars = sorted(list(set(''.join(words))))

In [36]:
stoi = {s : i for i,s in enumerate(chars)}
stoi # it basiucally giving a number to every character but we have to give number to out special characters
stoi['<S>'] = 26
stoi['<E>'] = 27
stoi

{'a': 0,
 'b': 1,
 'c': 2,
 'd': 3,
 'e': 4,
 'f': 5,
 'g': 6,
 'h': 7,
 'i': 8,
 'j': 9,
 'k': 10,
 'l': 11,
 'm': 12,
 'n': 13,
 'o': 14,
 'p': 15,
 'q': 16,
 'r': 17,
 's': 18,
 't': 19,
 'u': 20,
 'v': 21,
 'w': 22,
 'x': 23,
 'y': 24,
 'z': 25,
 '<S>': 26,
 '<E>': 27}

In [37]:
# now we can populate the n and map the characters to their specific numbers

for w in words:
  chs = ['<S>'] + list(w) + ['<E>']
  for ch1,ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1



In [ ]:
# lets visualize N using matplot lib